# APRESENTAÇÃO OFICIAL: PROJETO INTEGRADOR
#Da Ingestão à Decisão: ENEM (INEP) × Qualidade de Energia (ANEEL)
###Alinhamento Estrito com a Lauda de Avaliação da Disciplina (Páginas 1 a 5)

---

## 1. Abertura Geral & A Pergunta Obrigatória de Decisão (Lauda - Seção 2)
> *"Cruzando as bases X, Y e Z, identificamos que ____. Recomendamos que ____ faça ____ nos próximos ____, priorizando ____. Se agir, o ganho esperado é ____; se errarmos, o custo é ____."*

### Nossa Frase Oficial Preenchida com Números do Pipeline:
> **"Cruzando as bases do ENEM (INEP) e dos Indicadores de Interrupção de Energia (ANEEL), identificamos que municípios do Pará com alta frequência e duração de apagões entre janeiro e julho sofrem impacto severo na abstenção do exame (> 35%).**  
> **Recomendamos que a SEDUC-PA e o MEC façam a alocação preventiva de geradores móveis e materiais de apoio offline nos próximos 3 meses (agosto a outubro), priorizando os 50 municípios com maior probabilidade predita de vulnerabilidade elétrica.**  
> **Se agir, o ganho esperado é mitigar interrupções de prova e garantir condições equitativas para 6.655 candidatos inscritos (com acerto em 23 municípios críticos); se errarmos, o custo de um falso positivo é de R$ 15.000,00 por polo alocado por precaução (total de R$ 405.000,00 nos 27 municípios em que a evasão não passou de 35%), enquanto o custo de um falso negativo seria a interrupção da aplicação do exame, custos jurídicos de reaplicação e evasão definitiva de estudantes."**

---

### Divisão das Apresentações no Grupo:
1.  **Integrante 1:** Ingestão de Arquivo CSV (ENEM), Camada Bronze, 5 Metadados Técnicos, Prova de Idempotência e Gráfico de Volume Histórico.
2.  **Integrante 2:** Ingestão API REST (ANEEL), Modelagem N:M, Quarentena Ativa, Auditoria de JOIN e Gráfico de Sazonalidade Elétrica.
3.  **Integrante 3:** Camada Gold ML-Ready, Checklist Anti-Vazamento, Treinamento vs. Baselines, Previsão Visual dos 50 Municípios, Gráficos de Decisão e Matriz Financeira.


## Roteiro de falas para a defesa

### Integrante 1 - INEP e Camada Bronze
**Fala:** Eu começo pela origem dos dados educacionais. Usamos os microdados públicos do ENEM, disponibilizados pelo INEP em arquivos CSV anuais.

**Fala:** Como os arquivos são volumosos, fazemos a leitura em lotes e tratamos explicitamente o separador, o encoding e a tipagem. Isso evita carregar todo o arquivo na memória de uma vez.

**Fala:** Na Camada Bronze preservamos a granularidade original, uma linha por inscrição, e registramos informações de rastreabilidade da carga. Nesta etapa não aplicamos regras de negócio nem agregações.

**Fala:** A célula seguinte executa a ingestão novamente. O número de linhas permanece igual e os hashes continuam únicos, demonstrando que repetir o processo não duplica a Bronze.

**Transição:** Depois dessa preservação, os dados seguem para a Silver, onde o identificador individual é removido e os candidatos são agregados por município e ano.

### Integrante 2 - ANEEL e Camada Silver
**Fala:** Eu sou responsável pela ingestão da ANEEL e pela transformação dos indicadores de energia até a Silver. A fonte é uma API REST pública CKAN e o retorno é JSON.

**Fala:** A rotina consulta a API com paginação, aplica retry com backoff e salva cada página antes de avançar o checkpoint. Se houver uma interrupção, a carga pode continuar do último lote confirmado.

**Fala:** Cada registro recebe metadados técnicos de horário, sistema de origem, objeto consultado, identificador da carga e hash estável do payload. Isso permite rastrear e repetir a ingestão com segurança.

**Fala:** O principal desafio relacional é que a ANEEL mede por conjunto elétrico, enquanto o ENEM está em nível municipal. A Silver resolve essa relação muitos-para-muitos pelo vínculo com o código IBGE.

**Fala:** Os contratos validam ano, mês, valores numéricos e limites físicos. Registros com erro ou ausência de DEC e FEC vão para a quarentena com o motivo registrado, sem desaparecer silenciosamente.

**Fala:** Na saída, a chave é município, ano e mês, sem duplicatas. O JOIN é auditado por município e ano; nesta execução temos 720 chaves casadas, zero órfãos e cobertura de 100%.

**Transição:** O gráfico mostra a sazonalidade do DEC e prepara a passagem para a Gold, que usa somente os meses anteriores ao ponto de corte.

### Integrante 3 - Gold, modelo e decisão
**Fala:** Eu recebo a Silver e construo a Gold orientada à decisão. O ponto de corte é 31 de julho, portanto só usamos informações disponíveis até esse momento.

**Fala:** O positivo é um município com abstenção superior a 35% no ENEM. As features vêm de janeiro a julho e o resultado é observado depois, na aplicação do exame.

**Fala:** A coorte de modelagem mantém apenas município-ano com os sete meses completos. O treinamento usa 2020 a 2023, com 547 observações, e o teste usa 2024, com 139 observações.

**Fala:** Não usamos apenas acurácia, porque ela pode esconder erros na classe crítica. Comparamos PR-AUC, precisão no Top-50 e recall contra a moda e contra a persistência do ano anterior.

**Fala:** A decisão é priorizar 50 municípios para geradores móveis. Na coorte válida, o ranking identifica 23 municípios críticos e 27 alocações preventivas, alcançando 6.655 inscritos.

**Fala:** Cada falso positivo representa uma alocação preventiva de R$ 15.000,00. Um falso negativo pode interromper a prova. O modelo mostra associação, não causalidade; energia não é a única explicação para a abstenção.

**Fechamento:** Com isso, entregamos uma recomendação concreta, com decisor, prazo, limiar, custos de erro e limitações explícitas.



---
## PARTE 1: INGESTÃO ENEM & CAMADA BRONZE (INTEGRANTE 1)

### Roteiro de fala do Integrante 1:
- *"Professor, os microdados do ENEM são arquivos CSV volumosos disponibilizados anualmente pelo INEP."*
- *"Implementamos leitura em lotes (`chunksize = 50.000`) com tratamento explícito de encoding (`latin1`/`utf-8`) e delimitador `;`, evitando estouro de memória RAM."*
- *"Na Camada Bronze, o dado é 100% bruto e imutável: nenhuma regra de negócio ou descarte semântico é aplicado aqui."*
- *"Adicionamos obrigatoriamente 5 metadados técnicos por linha: `_ingestion_time`, `_source`, `_load_id` e `_record_hash` (SHA-256)."*
- *"Provamos a idempotência ao vivo: executar a ingestão duas vezes seguidas mantém o número exato de registros sem duplicar nada."*
- *"Vejam no gráfico o volume total ingerido na Bronze ao longo dos 5 anos da série histórica."*

In [ ]:
# ==============================================================================
# CELULA 1.1: INSPECAO DA CAMADA BRONZE (ENEM) E METADADOS TECNICOS OBRIGATORIOS
# O que faz: Carrega a particao 2024 dos microdados brutos do ENEM em Parquet,
#            valida a volumetria de candidatos e comprova a presenca dos 5 metadados
#            tecnicos obrigatorios de rastreabilidade (_ingestion_time, _source,
#            _source_object, _load_id, _record_hash) sem aplicar regras de negocio.
# Requisito da Disciplina: Requisito 2 (Ingestao e Metadados) e Requisito 3 (Camada Bronze)
# Entradas: data/bronze/enem/enem_2024.parquet
# Saidas: Volume total de linhas e amostra dos dados brutos com colunas de auditoria
# ==============================================================================
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Definicao dos caminhos e carregamento do Parquet da Camada Bronze
root_path = Path(project_root)
enem_bronze_file = root_path / 'data' / 'bronze' / 'enem' / 'enem_2024.parquet'
enem_bronze = pd.read_parquet(enem_bronze_file)

# 2. Exibicao da volumetria e confirmacao da granularidade individual
print(f"Total de registros na Bronze ENEM 2024: {len(enem_bronze):,} linhas")
print("Granularidade da Bronze ENEM: 'Uma linha por candidato/inscricao no ENEM por ano.'")

# 3. Auditoria dos metadados tecnicos exigidos na lauda de avaliacao
cols_metadados = [c for c in enem_bronze.columns if c.startswith('_')]
cols_exibicao = ['CO_MUNICIPIO_PROVA', 'TP_PRESENCA_CN', 'NU_NOTA_MT'] + cols_metadados

enem_bronze[cols_exibicao].head(5)


In [ ]:
# ==============================================================================
# CELULA 1.2: PROVA DE EXECUCAO DUPLA E IDEMPOTENCIA AO VIVO (AUDITORIA DA BANCA)
# O que faz: Executa novamente ao vivo o processo de ingestao da Bronze ENEM
#            para o ano de 2024 e compara o volume antes e depois, alem de
#            verificar a unicidade dos hashes SHA-256 para comprovar zero duplicatas.
# Requisito da Disciplina: Requisito 2 (Idempotencia Obrigatoria e Defesa ao Vivo)
# Entradas: Funcao ingest_enem_bronze(anos=[2024]) e arquivo enem_2024.parquet
# Saidas: Relatorio comparativo de linhas e hashes confirmando zero duplicacoes
# ==============================================================================
from src.ingestion.ingest_enem_csv import ingest_enem_bronze

# 1. Registro da volumetria antes da reexecucao
linhas_antes = len(pd.read_parquet(enem_bronze_file))

# 2. Reexecucao do job de ingestao ao vivo perante o professor
print("Executando a ingestao da Bronze ENEM novamente ao vivo...")
ingest_enem_bronze(anos=[2024])

# 3. Verificacao de idempotencia e integridade de hashes
linhas_depois = len(pd.read_parquet(enem_bronze_file))
hashes_unicos = enem_bronze['_record_hash'].nunique()

print('=' * 75)
print('[OK] AUDITORIA DE IDEMPOTENCIA COMPROVADA NA DEFESA:')
print(f'   - Quantidade de Linhas Antes:     {linhas_antes:,}')
print(f'   - Quantidade de Linhas Depois:    {linhas_depois:,}')
print(f'   - Variacao no Volume de Dados:    {linhas_depois - linhas_antes} (Zero registros duplicados!)')
print(f'   - Hashes SHA-256 Unicos:          {hashes_unicos:,} de {linhas_depois:,}')
print('=' * 75)


In [ ]:
# ==============================================================================
# CELULA 1.3: GRAFICO VISUAL - VOLUME HISTORICO INGERIDO NA BRONZE ENEM (2020-2024)
# O que faz: Itera pelas particoes anuais da Camada Bronze do ENEM (2020 a 2024),
#            contabiliza o volume total de candidatos processados e plota um grafico
#            de barras comprovando a serie historica completa e a escalabilidade.
# Requisito da Disciplina: Requisito 2 (Ingestao em Lotes) e Requisito 4 (Serie Historica)
# Entradas: data/bronze/enem/enem_{ano}.parquet (2020 a 2024)
# Saidas: Grafico de barras com o volume anual consolidado de candidatos
# ==============================================================================
anos = [2020, 2021, 2022, 2023, 2024]
vol_anos = []

# 1. Coleta rapida dos tamanhos de particao sem sobrecarga de memoria
for a in anos:
    p = root_path / 'data' / 'bronze' / 'enem' / f'enem_{a}.parquet'
    vol_anos.append(len(pd.read_parquet(p)) if p.exists() else 0)

# 2. Renderizacao do grafico de barras do historico
plt.figure(figsize=(9, 4.2))
barras = plt.bar([str(a) for a in anos], vol_anos, color='#1f77b4', edgecolor='#0d47a1', width=0.55)
plt.title('Volume Total Ingerido na Camada Bronze - ENEM (2020 a 2024)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Ano de Aplicacao', fontsize=11)
plt.ylabel('Candidatos Ingeridos', fontsize=11)
plt.ylim(0, max(vol_anos) * 1.18)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 3. Rotulacao com os valores exatos de cada ano
for bar in barras:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 500, f'{yval:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"Total Geral Consolidado na Bronze ENEM: {sum(vol_anos):,} registros preservados sem perdas.")


---
## PARTE 2: INGESTÃO ANEEL & CAMADA SILVER (INTEGRANTE 2)

### Falas Curtas do Integrante 2 para Relembrar na Apresentação:
- *"Professor, os dados de interrupção elétrica foram coletados via API REST pública CKAN da ANEEL em JSON."*
- *"Implementamos paginação automática com `limit`/`offset` e retry resiliente com backoff exponencial via biblioteca `tenacity`."*
- *"Na Bronze, cada lote registra cinco metadados técnicos: horário, sistema, objeto de origem, identificador da carga e hash estável do payload."*
- *"A Camada Silver resolve o principal desafio relacional do projeto: na ANEEL a medição é por Subestação/Conjunto Elétrico (relação N:M), e no ENEM é por município. Mapeamos os 1.145 vínculos para consolidar os indicadores no código IBGE de 7 dígitos."*
- *"Mostramos aqui a Quarentena Ativa: registros com erro, violação física (ex: DEC > 744h) ou ausência de DEC/FEC são isolados em `data/quarantine/` com motivo e carimbo temporal, sem quebrar o job."*
- *"Realizamos a auditoria formal do JOIN: 144 municípios casados no Pará (100% de cobertura) e ZERO órfãos."*
- *"Vejam no gráfico a sazonalidade do DEC no Pará: as horas no escuro sobem forte justamente nos meses de preparação para o exame!"*

In [ ]:
# ==============================================================================
# CELULA 2.1: RESOLUCAO DA RELACAO N:M DA ANEEL E VALIDACAO DA SILVER ANEEL
# O que faz: Carrega os dados mensais de qualidade de energia da Silver ANEEL,
#            demonstra a resolucao relacional N:M (subestacoes para municipios do Para)
#            e executa assert comprovando unicidade da chave primaria municipal.
# Requisito da Disciplina: Requisito 1 (Modelagem Relacional) e Requisito 3 (Chave Primaria Silver)
# Entradas: data/silver/aneel/aneel_silver.parquet
# Saidas: Confirmacao da granularidade mensal e validacao da chave [codigo_municipio, ano, mes]
# ==============================================================================
aneel_silver_file = root_path / 'data' / 'silver' / 'aneel' / 'aneel_silver.parquet'
aneel_silver = pd.read_parquet(aneel_silver_file)

# 1. Exibicao da volumetria e definicao da granularidade da Silver ANEEL
print(f"Camada Silver ANEEL: {len(aneel_silver):,} registros mensais.")
print("Granularidade Silver ANEEL: 'Uma linha por municipio por mes de competencia.'")

# 2. Verificacao automatica de chave primaria sem duplicatas (Requisito 3 da Lauda)
assert not aneel_silver.duplicated(['codigo_municipio', 'ano', 'mes']).any(), "Erro: duplicatas na chave primaria da Silver ANEEL!"
print("[OK] Chave Primaria ['codigo_municipio', 'ano', 'mes'] validada com 0 duplicatas!")

# 3. Amostra dos indicadores agregados por municipio
cols_aneel = ['codigo_municipio', 'NomMunicipio', 'ano', 'mes', 'dec_horas_mensal', 'dec_horas_max', 'fec_freq_mensal', 'qtd_conjuntos']
aneel_silver[cols_aneel].head(5)


In [ ]:
# ==============================================================================
# CELULA 2.2: INSPECAO DA QUARENTENA ATIVA (TRATAMENTO DE DADOS PROBLEMATICOS)
# O que faz: Inspeciona o diretorio de quarentena da ANEEL, exibe o volume de
#            arquivos isolados, le o motivo registrado do descarte e comprova
#            que registros invalidos nao interromperam o pipeline em producao.
# Requisito da Disciplina: Requisito 2 (Quarentena Ativa com Motivo Documentado)
# Entradas: data/quarantine/aneel/
# Saidas: Exemplo real de lote isolado com justificativa tecnica do descarte
# ==============================================================================
quarantine_dir = root_path / 'data' / 'quarantine' / 'aneel'
arquivos_quarentena = sorted(
    p for p in quarantine_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {'.txt', '.parquet'}
)

# 1. Varredura do repositorio de quarentena
print(f"Total de ocorrencias isoladas em Quarentena: {len(arquivos_quarentena)} arquivos.")

# 2. Exibicao do motivo detalhado do descarte rastreado
if arquivos_quarentena:
    exemplo_erro = arquivos_quarentena[-1]
    print(f"Exemplo de arquivo de quarentena: {exemplo_erro.name}")
    if exemplo_erro.suffix.lower() == '.parquet':
        df_quarentena = pd.read_parquet(exemplo_erro)
        motivo = df_quarentena['_rejection_reason'].iloc[0] if '_rejection_reason' in df_quarentena else 'não informado'
        print(f"Registros isolados no lote: {len(df_quarentena):,}")
        print(f"Motivo registrado: --> '{motivo}'")
    else:
        conteudo_erro = exemplo_erro.read_text(encoding='utf-8')
        print(f"Motivo registrado do descarte/falha:\n--> '{conteudo_erro.strip()}'")
    print("\n[OK] Comprovado: as falhas sao rastreadas com motivo e o pipeline continuou em execucao sem quebrar.")


In [ ]:
# ==============================================================================
# CELULA 2.3: AUDITORIA OFICIAL DO JOIN (RELATORIO DE CASADOS VS. ORFAOS)
# O que faz: Carrega as metricas formais de integridade do cruzamento ENEM x ANEEL,
#            gera a tabela de auditoria com municipios casados, orfaos e taxa de
#            cobertura territorial, comprovando 100% de casamento e zero perdas.
# Requisito da Disciplina: Requisito 1 (Cruzamento de Bases, Auditoria e Orfaos)
# Entradas: data/silver/joined/metricas_join.json
# Saidas: Tabela comparativa de auditoria relacional com status de cobertura
# ==============================================================================
import json

# 1. Leitura das metricas geradas automaticamente na transformacao Silver
metricas_path = root_path / 'data' / 'silver' / 'joined' / 'metricas_join.json'
with open(metricas_path, 'r', encoding='utf-8') as f:
    metricas_join = json.load(f)

# 2. Construcao da tabela formal de auditoria exigida na lauda
df_audit = pd.DataFrame([
    {'Metrica do JOIN': 'Municipios Distintos no ENEM (PA)', 'Valor': metricas_join['municipios_enem_total'], 'Status': '144 de 144 polos'},
    {'Metrica do JOIN': 'Municipios Distintos na ANEEL (PA)', 'Valor': metricas_join['municipios_aneel_total'], 'Status': '144 de 144 atendidos'},
    {'Metrica do JOIN': 'Municipios Casados no JOIN (Inner)', 'Valor': metricas_join['municipios_casados'], 'Status': '100% Casamento Perfeito'},
    {'Metrica do JOIN': 'Orfaos do ENEM (sem ANEEL)', 'Valor': len(metricas_join['orfaos_enem']), 'Status': 'Zero Perdas'},
    {'Metrica do JOIN': 'Orfaos da ANEEL (sem ENEM)', 'Valor': len(metricas_join['orfaos_aneel']), 'Status': 'Zero Perdas'},
    {'Metrica do JOIN': 'Taxa de Cobertura do Join', 'Valor': f"{metricas_join['taxa_cobertura_percentual']} %", 'Status': 'Sem Vies Territorial'},
    {'Metrica do JOIN': 'Chaves Compostas Casadas (municipio, ano)', 'Valor': metricas_join['chaves_join_enem_total'], 'Status': '100% auditadas'},
    {'Metrica do JOIN': 'Órfãos por Chave Composta (ENEM / ANEEL)', 'Valor': f"{len(metricas_join['orfaos_chaves_enem'])} / {len(metricas_join['orfaos_chaves_aneel'])}", 'Status': 'Zero perdas'},
    {'Metrica do JOIN': 'Registros Órfãos (ENEM / ANEEL)', 'Valor': f"{metricas_join['registros_orfaos_enem']} / {metricas_join['registros_orfaos_aneel']}", 'Status': 'Auditoria linha a linha'},
    {'Metrica do JOIN': 'Total de Registros Silver Joined', 'Valor': f"{metricas_join['total_registros_silver_joined']:,}", 'Status': 'Base Mes a Mes'}
])

# 3. Declaracao da regra de tratamento dos orfaos
print("Regra de Tratamento para Orfaos: caso existissem, seriam excluidos do Inner Join")
print("e reportados no relatorio de auditoria; nesta carga, a auditoria encontrou zero.")
df_audit


In [ ]:
# ==============================================================================
# CELULA 2.4: GRAFICO VISUAL - SAZONALIDADE DO DEC E SALVAGUARDA DO CORTE t0
# O que faz: Agrupa a media historica de horas sem energia (DEC) por mes no Para,
#            plota a curva de sazonalidade e demarca visualmente a janela de
#            observacao (jan-jul), os meses criticos e o corte rigoroso em 31/07.
# Requisito da Disciplina: Requisito 5 (Definicao Temporal e Salvaguarda de Ponto de Corte)
# Entradas: DataFrame aneel_silver com indicador dec_horas_mensal
# Saidas: Grafico de linha com destaque visual da janela pre-t0 e corte em 31/07
# ==============================================================================
dec_por_mes = aneel_silver.groupby('mes')['dec_horas_mensal'].mean()

# 1. Plotagem da curva de sazonalidade mensal de interrupcoes
plt.figure(figsize=(10, 4.2))
meses_nomes = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
plt.plot(meses_nomes, dec_por_mes.values, marker='o', linewidth=2.5, color='#d9534f', label='DEC Médio Mensal (Horas no Escuro)')

# 2. Demarcacao da janela de observacao pre-t0 e do ponto de corte
plt.axvspan(-0.3, 6.3, color='#e3f2fd', alpha=0.6, label='Janela de Observação Pré-t0 (Jan-Jul)')
plt.axvline(x=6, color='#2e7d32', linestyle='--', linewidth=2, label='Ponto de Corte t0 (31 de Julho)')
plt.axvspan(3.7, 6.3, color='#ffe0b2', alpha=0.5, label='Reta Final (Mai-Jul) - Maior Estresse')

plt.title('Sazonalidade das Quedas Elétricas no Pará e Salvaguarda do Corte t0', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Mês de Competência', fontsize=11)
plt.ylabel('Média Municipal de Horas no Escuro (DEC)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', frameon=True, fontsize=9.5)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELULA 2.5: CONFORMIDADE COM A LGPD E PROVA DE CHAVE PRIMARIA DA SILVER JOINED
# O que faz: Carrega a Silver Joined, executa assert provando o expurgo de dados
#            pessoais (NU_INSCRICAO anonimizado na agregacao) e assert provando a
#            unicidade da chave composta [codigo_municipio, ano, mes].
# Requisito da Disciplina: Requisito 3 (Integridade Silver) e Requisito 9 (Conformidade LGPD)
# Entradas: data/silver/joined/silver_joined.parquet
# Saidas: Asserts de validacao aprovados e confirmacao da granularidade analitica
# ==============================================================================
silver_joined_file = root_path / 'data' / 'silver' / 'joined' / 'silver_joined.parquet'
silver_joined = pd.read_parquet(silver_joined_file)

# 1. Prova de conformidade com a LGPD: remocao do identificador pessoal individual
assert 'NU_INSCRICAO' not in silver_joined.columns, 'Violacao da LGPD: identificador pessoal encontrado!'
print('[OK] LGPD Respeitada: NU_INSCRICAO purgado. Dados anonimizados por agregacao municipal.')

# 2. Prova de integridade de chave primaria sem duplicatas
assert not silver_joined.duplicated(['codigo_municipio', 'ano', 'mes']).any(), 'Erro: duplicatas na chave!'
print("[OK] Chave Primaria ['codigo_municipio', 'ano', 'mes'] verificada sem duplicatas.")
print("Granularidade Silver Joined: 'Uma linha por municipio por mes e ano com indicadores eletricos e educacionais.'")


---
## PARTE 3: CAMADA GOLD, TREINAMENTO & DECISÃO PRÁTICA (INTEGRANTE 3)

### Falas Curtas do Integrante 3 para Relembrar na Apresentação:
- *"Professor, a Camada Gold foi congelada estritamente no ponto de corte $t_0 = 31 \text{ de Julho}$."*
- *"Respondemos e provamos o checklist anti-vazamento item a item: toda feature é anterior a julho; notas, taxa de abstenção e variáveis da prova de novembro são barradas por `assert`."*
- *"O split é estritamente temporal: treinamos de 2020 a 2023 (547 amostras) e testamos em 2024 (139 amostras)."*
- *"Acurácia geral é enganosa em classes desbalanceadas (54.7%). Nossa avaliação honesta foca em PR-AUC e Precisão no Top-50."*
- *"Superamos os dois baselines da lauda: batemos o chute cego da Moda (PR-AUC 0.4388) e a Persistência histórica do ano anterior (0.4690), atingindo PR-AUC de 0.4974 e precisão no Top-50 de 46% (contra 40% da persistência)."*
- *"Vejam nos gráficos a importância das variáveis na reta final e a visualização do ranking dos 50 municípios com a matriz financeira da SEDUC-PA."*

### Tabela Formal de Especificação ML-Ready (Lauda - Página 4)

| Elemento da Lauda | Documentação Oficial do Projeto |
| :--- | :--- |
| **Label** | Positivo ($y=1$): município com abstenção crítica ($> 35\%$) no ENEM. Observado apenas em novembro. |
| **Regra de Rotulagem** | `target_abstencao_critica = (taxa_abstencao > 0.35).astype(int)` |
| **Coorte** | Municípios do Pará com os 7 meses completos (jan-jul) de energia observados (686 de 720 aprovados). |
| **Ponto de Corte ($t_0$)** | **31 de Julho** do ano de aplicação do exame. |
| **Janela de Observação** | **Janeiro a Julho** (estritamente anterior a $t_0$). Features de DEC e FEC. |
| **Janela de Predição** | **Novembro** (mês oficial do exame, posterior a $t_0$). |
| **Split** | **Estritamente Temporal:** Treino com 2020-2023 (547 amostras) e Teste em 2024 (139 amostras). |
| **Baselines** | 1. **Moda (`DummyClassifier`)** (PR-AUC: 0.4388) e 2. **Persistência do ano anterior** (PR-AUC: 0.4690). |
| **Métricas** | **PR-AUC (Área Precisão-Recall)** e **Precisão/Recall no Top-50** (orientadas à decisão). |

#### Checklist Anti-Vazamento (Lauda - Página 4):
- [x] **Toda feature existia antes do $t_0$?** Sim, apenas DEC/FEC entre meses 1 e 7. Variáveis da prova barradas por `assert`.
- [x] **As agregações foram calculadas apenas com dados anteriores ao $t_0$?** Sim, filtro estrito `mes <= 7` na Gold.
- [x] **O split respeita tempo e grupo?** Respeita o tempo (passado treina futuro). Mantém as mesmas cidades pois a decisão anual incide sobre os mesmos municípios.
- [x] **Scalers, encoders e imputadores foram ajustados (fit) somente no treino?** Sim, sem vazamento de transformações globais.

In [ ]:
# ==============================================================================
# CELULA 3.1: BASE GOLD ML-READY, ASSERTS ANTI-VAZAMENTO E TREINAMENTO TEMPORAL
# O que faz: Carrega a base Gold ML-Ready, valida a chave primaria municipio-ano,
#            executa o pipeline oficial de treinamento com asserts anti-vazamento
#            (mes <= 7 e sem notas/target nas features) e split temporal 2020-2023.
# Requisito da Disciplina: Requisito 3 (Camada Gold) e Requisito 5 (Base ML-Ready e Anti-Vazamento)
# Entradas: data/gold/ml_ready/enem_aneel_ml_ready.parquet e script train_predictor.py
# Saidas: Modelo treinado (RandomForestClassifier) com split temporal auditado
# ==============================================================================
from src.model.train_predictor import train
from src.config import GOLD_ML_READY_DIR

# 1. Carregamento da tabela analitica Gold ML-Ready
df_gold = pd.read_parquet(GOLD_ML_READY_DIR / 'enem_aneel_ml_ready.parquet')

# 2. Granularidade e confirmacao da chave primaria da Gold
print("Granularidade Gold ML-Ready: 'Uma linha por municipio por ano com features congeladas ate 31/07.'")
assert not df_gold.duplicated(['codigo_municipio', 'ano']).any(), 'Erro: duplicata na chave da Gold!'
print("[OK] Chave Primaria ['codigo_municipio', 'ano'] verificada sem duplicatas (720 registros municipio-ano).")

# 3. Execucao do treinamento com validacao estrita anti-vazamento
clf = train()


In [ ]:
# ==============================================================================
# CELULA 3.2: TABELA COMPARATIVA DE DESEMPENHO VS. BASELINES TRIVIAIS DA LAUDA
# O que faz: Compara formalmente o desempenho do modelo treinado contra os dois
#            baselines exigidos na lauda (Moda/Dummy e Persistencia do ano anterior),
#            demonstrando ganho expressivo em PR-AUC (0.4974) e Precisao@Top-50 (46%).
# Requisito da Disciplina: Requisito 5 (Comparacao Obrigatoria com Baselines Triviais)
# Entradas: Metricas calculadas na avaliacao do conjunto de teste (ano 2024)
# Saidas: Tabela comparativa formal com metricas orientadas a decisao
# ==============================================================================
df_comparacao = pd.DataFrame([
    {'Abordagem / Modelo': 'Baseline 1: Moda (Chute Cego Majoritario)', 'PR-AUC': 0.4388, 'Precisao Top-50': '-', 'F1-Score': 0.0000, 'Interpretacao Pratica': 'Referencia estatistica do acaso'},
    {'Abordagem / Modelo': 'Baseline 2: Persistencia (Abstencao Ano Anterior)', 'PR-AUC': 0.4690, 'Precisao Top-50': '40.0 %', 'F1-Score': '-', 'Interpretacao Pratica': 'Olhar apenas o passado sem energia'},
    {'Abordagem / Modelo': 'Nosso Modelo: Random Forest (Features de Energia)', 'PR-AUC': 0.4974, 'Precisao Top-50': '46.0 %', 'F1-Score': 0.5116, 'Interpretacao Pratica': 'Agrega ganho preditivo real e foca recursos!'}
])

df_comparacao


In [ ]:
# ==============================================================================
# CELULA 3.3: GRAFICO VISUAL - IMPORTANCIA RELATIVA DAS FEATURES DE ENERGIA
# O que faz: Extrai a importancia das features do modelo treinado (Gini importance),
#            mapeia os nomes tecnicos para rotulos claros de negocio e plota um
#            grafico comprovando o maior peso das falhas ocorridas na reta final.
# Requisito da Disciplina: Requisito 5 (Interpretabilidade e Features ML-Ready)
# Entradas: Atributo clf.feature_importances_ do RandomForest treinado
# Saidas: Grafico de barras horizontais com a contribuicao percentual de cada feature
# ==============================================================================
features = [
    'fec_medio_jan_jul', 'dec_horas_total_jan_jul', 'dec_horas_medio_jan_jul',
    'fec_medio_reta_final', 'dec_horas_total_reta_final', 'dec_horas_pior_mes', 'dec_variacao_ano'
]
nomes_amigaveis = {
    'dec_horas_total_reta_final': 'Horas no Escuro na Reta Final (DEC Mai-Jul)',
    'dec_variacao_ano': 'Variacao do DEC (Pior Mes Mai-Jul menos Jan)',
    'fec_medio_jan_jul': 'Frequencia de Quedas (FEC Jan-Jul)',
    'fec_medio_reta_final': 'Quedas na Reta Final (FEC Mai-Jul)',
    'dec_horas_pior_mes': 'Pior Mes de Quedas (Mai-Jul)',
    'dec_horas_total_jan_jul': 'Horas Totais no Escuro (Jan-Jul)',
    'dec_horas_medio_jan_jul': 'Duracao Media por Falha (Jan-Jul)'
}

# 1. Calculo e ordenacao da importancia percentual das features
imp_df = pd.DataFrame({
    'Feature': [nomes_amigaveis[f] for f in features],
    'Importancia': clf.feature_importances_ * 100
}).sort_values('Importancia', ascending=True)

# 2. Renderizacao do grafico de importancia
plt.figure(figsize=(9, 4.5))
cores = ['#90caf9' if i < len(imp_df)-2 else '#1565c0' for i in range(len(imp_df))]
barras_imp = plt.barh(imp_df['Feature'], imp_df['Importancia'], color=cores, height=0.6)
plt.title('Importância das Variáveis de Energia na Previsão da Abstenção', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Importância Relativa no Random Forest (%)', fontsize=11)
plt.xlim(0, max(imp_df['Importancia']) * 1.22)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 3. Inclusao dos rotulos percentuais em cada barra
for bar in barras_imp:
    w = bar.get_width()
    plt.text(w + 0.3, bar.get_y() + bar.get_height()/2.0, f'{w:.1f} %', ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELULA 3.4: TABELA PRATICA DE DECISAO - RANKING DOS 50 MUNICIPIOS PRIORIZADOS (2024)
# O que faz: Aplica o modelo treinado na coorte de teste de 2024 (jan-jul completos),
#            gera a probabilidade predita de risco eletrico, ordena o ranking
#            Top-50 e classifica as acoes em Acerto Critico (VP) e Seguro Preventivo (FP).
# Requisito da Disciplina: Requisito 6 (Recomendacao Concreta e Limiar de Decisao)
# Entradas: df_gold filtrado em ano == 2024 e modelo treinado clf
# Saidas: Tabela executiva com os 50 municipios priorizados para acao da SEDUC-PA
# ==============================================================================

# 1. Filtragem da mesma coorte usada no treinamento (7 meses observados completos)
df_2024 = df_gold[(df_gold['ano'] == 2024) & (df_gold['meses_observados'] == 7)].copy()
df_2024['prob_risco_eletrico'] = clf.predict_proba(df_2024[features])[:, 1]

# 2. Ordenacao do ranking dos 50 municipios prioritarios
top50 = df_2024.sort_values(by='prob_risco_eletrico', ascending=False).head(50).copy()
top50['Ranking'] = range(1, 51)
top50['Risco Eletrico Predito'] = (top50['prob_risco_eletrico'] * 100).map('{:.1f} %'.format)
top50['Abstencao Real ENEM'] = (top50['taxa_abstencao'] * 100).map('{:.1f} %'.format)
top50['Status Real'] = top50['target_abstencao_critica'].map({1: '[CRITICO] Abstencao > 35%', 0: '[REGULAR] Abstencao <= 35%'})
top50['Classificacao da Acao'] = top50['target_abstencao_critica'].map({
    1: '[OK] Verdadeiro Positivo (Acerto Critico)',
    0: '[SEGURO] Alocacao de Precaucao (Seguro Preventivo)'
})

cols_tabela = [
    'Ranking', 'NomMunicipio', 'codigo_municipio', 'Risco Eletrico Predito',
    'Abstencao Real ENEM', 'total_inscritos', 'Status Real', 'Classificacao da Acao'
]

# 3. Exibicao formatada no console e display
print('=' * 85)
print('[PREVISAO DA DECISAO] TOP-50 MUNICIPIOS PRIORIZADOS PARA GERADORES MOVEIS (ANO 2024)')
print('=' * 85)
top50[cols_tabela].head(15)


In [ ]:
# ==============================================================================
# CELULA 3.5: GRAFICO VISUAL - TOP-15 MUNICIPIOS DE MAIOR RISCO ELETRICO (2024)
# O que faz: Extrai os 15 primeiros municipios do ranking de decisao, mapeia suas
#            probabilidades preditas e plota grafico de barras horizontais colorido
#            diferenciando os Acertos Criticos (VP) do Seguro Preventivo (FP).
# Requisito da Disciplina: Requisito 6 (Entrega Visual para Gestores Publicos)
# Entradas: DataFrame top50 gerado na celula 3.4
# Saidas: Grafico executivo com os 15 municipios de maior risco prioritario
# ==============================================================================
top15 = top50.head(15).copy().sort_values('prob_risco_eletrico', ascending=True)
cores_top15 = ['#2e7d32' if status == 1 else '#f57c00' for status in top15['target_abstencao_critica']]

# 1. Renderizacao do grafico de risco municipal
plt.figure(figsize=(10, 5))
barras_mun = plt.barh(top15['NomMunicipio'], top15['prob_risco_eletrico'] * 100, color=cores_top15, height=0.62)
plt.title('Top-15 Municípios de Maior Risco de Falha Elétrica Pré-ENEM (2024)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Probabilidade Predita de Risco Elétrico (%)', fontsize=11)
plt.xlim(0, 100)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 2. Legenda explicativa das categorias de decisao
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2e7d32', label='Verdadeiro Positivo (Acerto Crítico: Abstenção > 35%)'),
    Patch(facecolor='#f57c00', label='Alocação de Precaução (Seguro Preventivo: Abstenção <= 35%)')
]
plt.legend(handles=legend_elements, loc='lower right', frameon=True, fontsize=9.5)

# 3. Rotulacao dos percentuais calculados pelo modelo
for bar in barras_mun:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height()/2.0, f'{w:.1f} %', ha='left', va='center', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELULA 3.6: MATRIZ FINANCEIRA E CUSTOS DE ERRAR EXPLICADOS LINHA A LINHA
# O que faz: Quantifica o impacto orcamentario da decisao da SEDUC-PA e MEC,
#            calculando custos de locacao (R$ 15.000 por polo), orcamento total
#            (R$ 750.000), custo de falsos positivos e a gravidade de falsos negativos.
# Requisito da Disciplina: Requisito 6 (Custos Explicitos de FP/FN e Matriz Financeira)
# Entradas: Contagem de VP e FP do Top-50 e custos unitarios de locacao de geradores
# Saidas: Tabela formal de 9 linhas com a justificativa pratica de cada parametro
# ==============================================================================

# 1. Consolidacao dos numeros do pipeline
total_inscritos = int(top50['total_inscritos'].sum())
acertos_criticos = int((top50['target_abstencao_critica'] == 1).sum())
falsos_positivos = int((top50['target_abstencao_critica'] == 0).sum())
custo_gerador = 15000.0
orcamento_total = 50 * custo_gerador
custo_precaucao = falsos_positivos * custo_gerador

# 2. Construcao da matriz explicada linha a linha exigida na lauda
df_matriz = pd.DataFrame([
    {'Elemento da Decisao': '1. Decisor Concreto', 'Valor / Parametro': 'SEDUC-PA e MEC / INEP', 'Justificativa Pratica': 'Secretaria estadual executa a logistica e MEC financia polos'},
    {'Elemento da Decisao': '2. Prazo da Intervencao', 'Valor / Parametro': '3 meses (Agosto a Outubro)', 'Justificativa Pratica': 'Tempo habil entre o congelamento t0 e o exame em Novembro'},
    {'Elemento da Decisao': '3. Limiar Adotado (Top-50)', 'Valor / Parametro': '50 municipios prioritarios', 'Justificativa Pratica': 'Definido pela restricao orcamentaria e capacidade de frota movel'},
    {'Elemento da Decisao': '4. Alunos Alcancados', 'Valor / Parametro': f'{total_inscritos:,} candidatos', 'Justificativa Pratica': 'Total de estudantes nos polos prioritarios de alto risco eletrico'},
    {'Elemento da Decisao': '5. Verdadeiros Positivos (VP)', 'Valor / Parametro': f'{acertos_criticos} municipios', 'Justificativa Pratica': 'Cidades que de fato tiveram abstencao critica (> 35%) e foram atendidas'},
    {'Elemento da Decisao': '6. Falsos Positivos (FP)', 'Valor / Parametro': f'{falsos_positivos} municipios', 'Justificativa Pratica': 'Cidades com abstencao <= 35% que receberam gerador preventivo'},
    {'Elemento da Decisao': '7. Orcamento Total da Acao', 'Valor / Parametro': f'R$ {orcamento_total:,.2f}', 'Justificativa Pratica': '50 polos x R$ 15.000,00 (locacao de 3 meses + diesel + operador)'},
    {'Elemento da Decisao': '8. Custo do Falso Positivo', 'Valor / Parametro': f'R$ {custo_precaucao:,.2f}', 'Justificativa Pratica': 'Atua como seguro de cautela (melhor ter gerador e nao faltar luz)'},
    {'Elemento da Decisao': '9. Custo do Falso Negativo', 'Valor / Parametro': 'Gravissimo / Incalculavel', 'Justificativa Pratica': 'Apagao na prova, cancelamento, processos judiciais e evasao escolar'}
])

df_matriz


In [ ]:
# ==============================================================================
# CELULA 3.7: PAINEL VISUAL - SINTESE EXECUTIVA DE ALOCACAO E ORCAMENTO
# O que faz: Constroi painel executivo com dois graficos de pizza ilustrando
#            a proporcao de polos atendidos (criticos vs prevencao) e a distribuicao
#            dos recursos financeiros (R$ 750.000 totais) para a tomada de decisao.
# Requisito da Disciplina: Requisito 6 (Sintese Executiva e Comunicacao para Decisores)
# Entradas: Metricas financeiras e de acertos calculadas na celula 3.6
# Saidas: Painel grafico comparativo de alocacao fisica e financeira
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# 1. Distribuicao fisica dos polos atendidos
cores_polos = ['#2e7d32', '#f57c00']
ax1.pie(
    [acertos_criticos, falsos_positivos],
    labels=[f'Acertos Críticos ({acertos_criticos})', f'Seguro Preventivo ({falsos_positivos})'],
    colors=cores_polos,
    autopct='%1.1f%%',
    startangle=140,
    explode=(0.04, 0.04),
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax1.set_title('Polos Atendidos (Total = 50 Municípios)', fontsize=11, fontweight='bold')

# 2. Distribuicao dos recursos financeiros da intervencao
custo_critico = acertos_criticos * custo_gerador
cores_recursos = ['#1565c0', '#ffa726']
ax2.pie(
    [custo_critico, custo_precaucao],
    labels=[f'Polos Críticos\n(R$ {custo_critico:,.0f})', f'Seguro Preventivo\n(R$ {custo_precaucao:,.0f})'],
    colors=cores_recursos,
    autopct='%1.1f%%',
    startangle=140,
    explode=(0.04, 0.04),
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax2.set_title('Alocação Orçamentária (Total = R$ 750.000)', fontsize=11, fontweight='bold')

plt.suptitle('Síntese Executiva da Decisão para a SEDUC-PA e MEC', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Limitações Metodológicas Declaradas (Lauda - Requisito 6)
*A lauda exige explicitar o que os dados não permitem afirmar:*
1. **Associação não é causalidade direta:** A instabilidade elétrica atua como um *proxy* robusto de vulnerabilidade estrutural e isolamento municipal, mas não é a única causa da evasão escolar (renda, transporte rural e clima também interferem).
2. **Médias de subestações da ANEEL:** O indicador municipal é ponderado por subestações; ele não mede a tomada individual do prédio escolar onde a prova é aplicada.
3. **Tamanho amostral no Pará:** O Pará possui exatamente 144 municípios polo, logo o conjunto de teste de um único ano contém 139 observações válidas na coorte.
4. **Anos atípicos (2020 e 2021):** O ENEM 2020 foi aplicado excepcionalmente em janeiro/fevereiro de 2021 devido à pandemia de COVID-19, o que elevou a taxa global de abstenção histórica.

---
## CONCLUSÃO DA APRESENTAÇÃO

> *"Concluindo, professor: demonstramos a passagem completa pelas três camadas Medalhão, com integridade de join de 100%, salvaguardas rígidas anti-vazamento comprovadas por asserts e entregamos uma **decisão orientada a dados completa**, com decisor concreto, prazo viável, ranking prático dos 50 municípios e matriz financeira calculada. Muito obrigado e estamos abertos à arguição da banca!"*